<a href="https://colab.research.google.com/github/paulsoriiiano/cmpe-259-project/blob/main/notebooks/parks_virtual_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# California Parks Virtual Assistant

### Final Project

Fall 2025

CMPE 259 - Natural Language Processing

Author: Paul Junver Soriano

## Description

smth...

## Project Set Up

In [1]:
# Get data and Python modules from repo
!git clone https://github.com/paulsoriiiano/cmpe-259-project
!mv cmpe-259-project/data .
!mv cmpe-259-project/src .
!rm -rf cmpe-259-project

Cloning into 'cmpe-259-project'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 224 (delta 121), reused 142 (delta 60), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 605.52 KiB | 15.93 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [3]:
# Install required libraries

%%bash
# Vectorstore libraries
pip install faiss-cpu jq

# Huggingface clients
pip install huggingface_hub

# Langchain dependencies
pip install langchain langchain-community langchain_core langchain-huggingface langchain-mistralai langchain-openai

# Weather tool dependencies
pip install geopy openmeteo_requests requests-cache retry-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.1/167.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.2/684.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 68.7 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login
from google.colab import userdata

login(new_session=False)
hf_token = userdata.get("HF_TOKEN")         # Get HuggingFace API Token

## Load models

In [5]:
from src.llm_utils import load_chat_model

small_llm = load_chat_model("small")
large_llm = load_chat_model("large")

In [ ]:
# Sample query
query = "What is the weather like this weekend in Antelope Valley?"

In [ ]:
print(small_llm.invoke(query).content)

 I cannot provide an exact answer without checking the current weather forecast for Antelope Valley. Generally, Antelope Valley in California experiences hot and dry weather during the summer months. However, I would recommend checking a reliable weather source for the most up-to-date and accurate information. You can visit the National Weather Service website or download a weather app to check the forecast for Antelope Valley specifically.


In [ ]:
print(large_llm.invoke(query).content)

I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways for you to find out the weather forecast for Antelope Valley this weekend.

You can check online weather websites such as:

1. National Weather Service (NWS): [www.weather.gov](http://www.weather.gov)
2. AccuWeather: [www.accuweather.com](http://www.accuweather.com)
3. Weather.com: [www.weather.com](http://www.weather.com)

You can also check mobile apps like Dark Sky or Weather Underground for hyperlocal weather forecasts.

Additionally, you can search for "Antelope Valley weather forecast" or " Lancaster, CA weather forecast" (if you're referring to the Antelope Valley in California) to get the latest weather updates.

Please note that weather forecasts are subject to change, so it's always a good idea to check the forecast again closer to the weekend for the most up-to-date information.


## Get vector database

In [6]:
from src.vector_db import build_vector_db, load_vector_db

try:
  retriever = load_vector_db().as_retriever()
except:
  retriever = build_vector_db().as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Build weather function

In [ ]:
import src.weather_fn
from src.weather_fn import get_weather

# Test weather function
print(get_weather("Calaveras Big Trees State Park", days=2))

2-day forecast for Calaveras Big Trees State Park:

Day 1: moderate snow
- Temp: 36.8 —— 44.1°F
- Precipitation: 1.39 inches
- Max Wind: 13.8 mph

Day 2: moderate snow
- Temp: 33.8 —— 40.8°F
- Precipitation: 0.17 inches
- Max Wind: 9.0 mph


## Build RAG chains

In [21]:
# """
# This module provides querying functions.
# """

# import re

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableLambda, RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser

# import src.weather_fn
# from src.weather_fn import get_weather


# def build_rag_chain(llm, retriever):
#   """ Build a two-step RAG chain. """

#   # 1. Retrieve documents
#   def retrieve_and_format(query):
#     # 1.1. Get documents context
#     rag_docs = retriever.invoke(query)
#     rag_context = "\n\n".join([d.page_content for d in rag_docs])
#     rag_sources = "\n\n".join([f"{d.metadata["title"]} information from {d.metadata["url"]}" for d in rag_docs])

#     # 1.2. Check if query requires weather information
#     keywords = ["weather", "temperature", "day", "rain", "sunny", "snow"]
#     if any(word in query.lower() for word in keywords):

#       # 1.3. Get location (if available)
#       location = rag_docs[0].metadata.get("title") if rag_docs else None
#       if not location:
#         rag_context += "\n\n No location found. Could not get weather data."

#       # 1.4. Get weather information (if available)
#       match = re.search(r"(\d+)[ -]?day", query)
#       days = int(match.group(1)) if match else 0
#       weather_info = get_weather(location, days)
#       if not weather_info:
#         rag_context += "\n\n No weather information found. Something went wrong with fetching weather data."

#       rag_context += f"\n\n Weather information: {weather_info}\n"

#     inputs = {
#         "context": rag_context,
#         "question": query,
#         "sources": rag_sources,
#     }

#     return inputs

#   # 2. Create prompt
#   prompt = ChatPromptTemplate.from_messages([
#         ("system",
#          "You are a helpful California Parks & Trails assistant."
#          "Answer concisely and cite the relevant parks information in your answer.",
#          ),

#         ("human",
#          """Use the following context and sources from park data to answer questions.
#             If information is missing, respond with the best estimate or say "I'm not sure."
#             Format the sources (if available) as footnotes in the end of the answer.

#             Context:
#             {context}

#             Sources:
#             {sources}

#             Question:
#             {question}

#             Your answer:

#           """
#         )
#     ])

#   # 3. Build chain
#   rag_chain = (
#     RunnableLambda(retrieve_and_format)
#     | prompt
#     | llm
#     | StrOutputParser()
#   )

#   return rag_chain

In [16]:
from src.rag_pipeline import build_rag_chain

small_rag_chain = build_rag_chain(small_llm, retriever)
large_rag_chain = build_rag_chain(large_llm, retriever)

In [20]:
# Test RAG chain.
query = "Is today a good day to go to Calaveras Big Trees?"

print(f"Small LLM response: \n\n{small_rag_chain.invoke(query)}")
print("\n======================================================\n")
print(f"Large LLM response: \n\n{large_rag_chain.invoke(query)}")

Small LLM response: 

 Yes, today is a good day to go to Monterey State Beach based on the current clear sky, a temperature of 54°F, and winds from the north-northwest at 9 mph with gusts up to 14 mph [1][2].

Sources:
[1] Monterey State Beach Current Weather from https://weather.com/us/ca/monterey/current-weather/135918
[2] Monterey State Beach Information from https://www.parks.ca.gov/?page_id=576


Large LLM response: 

Yes, today appears to be a good day to visit Monterey State Beach. The current weather is clear skies with a temperature of 54°F and moderate wind, which is suitable for outdoor activities like surfing, tidepool watching, and walking.¹

¹ https://www.parks.ca.gov/?page_id=576


## Build user prompts

## Agentic Approach

### Build tools

In [ ]:
import re
from langchain.tools import Tool

# RAG tool for different agents
small_rag_tool = Tool(
    name="parks_info",
    func=lambda q: small_rag_chain.invoke(q),
    description="Parks info via small LLM (Mistral-7B)"
)

large_rag_tool = Tool(
    name="parks_info",
    func=lambda q: large_rag_chain.invoke(q),
    description="Parks info via large LLM (Llama-3.3-70B)"
)

# Weather tool
def weather_wrapper(query: str):
  match = re.search(r"(\d+)[ -]?day", query)
  days = int(match.group(1)) if match else 0
  return get_weather(query, 0)

weather_tool = Tool(
    name="WeatherTool",
    func=weather_wrapper,
    description="Get weather info for California parks via OpenMeteo"
)

### Build agents

In [ ]:
from langchain.agents import initialize_agent, AgentType

# Group tools
small_tools = [small_rag_tool, weather_tool]
large_tools = [large_rag_tool, weather_tool]

# Initilize agents
small_agent = initialize_agent(
    small_tools,
    llm=small_llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True
)

large_agent = initialize_agent(
    large_tools,
    llm=large_llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True
)

### Build user queries

In [ ]:
small_rag_chain.invoke("Calaveras Big Trees")

' Calaveras Big Trees State Park undergoes seasonal closures. It is open for day use and camping from May 15th until Winter, though the specific dates may vary each year. The park features giant sequoias, pines, flowing rivers and creeks, wildlife, meadows, and offers camping, fishing, hiking, swimming, and more. Dogs are allowed only in the campgrounds and on fire roads. For more information or reservations, visitors can contact (209) 795-2334. The Visitor Center offers a museum, film, and gift shop. The South Grove Trail reopens every year around May 1st. The park permanently protects more than 150,000 acres in California State Parks redwood parks. For additional information, visit SaveTheRedwoods.org or contact the park through its Contact Us page.\n\nSources:\nCalaveras Big Trees State Park information from https://www.parks.ca.gov/?page_id=551\nCalifornia State Parks and CAL FIRE Plan Prescribed Burns at Calaveras Big Trees State Park (9/24/25), https://www.parks.ca.gov/news/25732

In [ ]:
q_info = "Tell me about Calaveras Big Trees."
q_weather = "What is the weather like in Calaveras Big Trees?"

In [ ]:
small_agent.run(q_info)

'Calaveras Big Trees State Park is a California state park that is open from sunrise to sunset for day use, and camping is available as of May 15th. The park allows dogs in campgrounds and on fire roads, and offers activities such as camping, fishing, hiking, and swimming. It protects giant sequoias and has a reopened South Grove Trail as of May 1, 2025. The park is not suitable for trailers and large motorhomes due to steep roads and winter closures. For more information, visit <https://www.parks.ca.gov/?page_id=551>.'

In [ ]:
large_agent.run(q_info)

'Calaveras Big Trees State Park is a California state park that features giant sequoias, pines, rivers, creeks, wildlife, and meadows, and offers activities such as camping, fishing, hiking, and swimming. The park has a Visitor Center with a museum, film, and gift shop, and dogs are allowed in campgrounds and on fire roads. Currently, the weather at the park is overcast with a temperature of 42°F, humidity of 88%, and wind of 6 mph, with no precipitation.'

### Test both agents on those queries